# 04 · ECGFounder Native 500 Hz Fine-Tuning

Part of the study *Multi-Label 12-Lead ECG Classification on the PTB-XL Dataset* ([arXiv:2609.12803](https://arxiv.org/abs/2609.12803)). Run order, folder layout, and results: [project README](../README.md).

> **GPU (Colab):** NVIDIA **A100** (40 GB). L4 is acceptable. If a T4 runs out of memory, set `batch_size` to 8.

Fine-tunes pretrained ECGFounder (Net1D) on official PTB-XL folds using **native 500 Hz** tensors from notebook 00 (`processed/sr500/`, shape `(N, 12, 5000)`). The pretrained backbone expects 5000 samples, so this notebook does **not** interpolate and does **not** re-read `filename_hr`.

Forward pass: per-record global z-score only. Full fine-tune: HuggingFace `PKUDigitalHealth/ECGFounder` backbone + a new 5-class `dense` head; the backbone is not frozen.

Shared protocol (`ecg_common.py`): official folds 1–8 / 9 / 10, checkpoint = validation **macro-AUPRC**, val F1-max thresholds, reported metrics = test fold 10.

Outputs: `experiments/ecgfounder_500/seed_<seed>/` and canonical results `results/ecgfounder/seed_<seed>/` (ensemble member key `ecgfounder`).


In [ ]:
!pip install -q openpyxl scikit-learn pandas numpy matplotlib seaborn torch tqdm huggingface_hub


## Foundation / path settings

`SAMPLING_RATE = 500` and `processed/sr500` are required. Notebooks 01–03 / 02.1 keep `sr100`. Run notebook 00 through the memmap export cell first.

Training hyperparameters (`batch_size`, `epochs`, `learning_rate`) are in the `ExperimentConfig` cell after the Model section.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')

SAMPLING_RATE = 500
MODEL_NAME = 'ecgfounder_500'
FOUNDATION_BACKEND = 'ecgfounder'
ECGFOUNDER_HF_REPO = 'PKUDigitalHealth/ECGFounder'
ECGFOUNDER_TARGET_LEN = 5000

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'ECG AI'
PROCESSED_DIR = PROJECT_DIR / 'processed' / f'sr{SAMPLING_RATE}'
EXPERIMENTS_DIR = PROJECT_DIR / 'experiments'
RESULTS_DIR = PROJECT_DIR / 'results'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import load_processed_manifest, project_paths

_paths = project_paths(PROJECT_DIR, sampling_rate=SAMPLING_RATE)
assert _paths['processed_dir'] == PROCESSED_DIR
_manifest = load_processed_manifest(PROCESSED_DIR)
print('processed:', PROCESSED_DIR)
print('dataset_version:', _manifest.get('dataset_version') or _manifest.get('raw_root_name'))
print('split:', _manifest.get('split_by'), _manifest.get('train_folds'), _manifest.get('val_folds'), _manifest.get('test_folds'))
print('manifest sampling_rate:', _manifest.get('sampling_rate'), '| n_samples', _manifest.get('n_samples'))
print('source:', _manifest.get('source'))
assert int(_manifest.get('sampling_rate', -1)) == 500, 'Need processed/sr500 from notebook 00'
assert int(_manifest.get('n_samples', -1)) == 5000, 'Need n_samples=5000 native 500 Hz tensors'


In [ ]:
# ECGFOUNDER_SETUP
import sys
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download

ECGFOUNDER_ROOT = Path('/content/ECGFounder')
if not ECGFOUNDER_ROOT.exists():
    subprocess.run(
        ['git', 'clone', '-q', 'https://github.com/PKUDigitalHealth/ECGFounder.git', str(ECGFOUNDER_ROOT)],
        check=True,
    )
if str(ECGFOUNDER_ROOT) not in sys.path:
    sys.path.insert(0, str(ECGFOUNDER_ROOT))

_finetune_py = ECGFOUNDER_ROOT / 'finetune_model.py'
_txt = _finetune_py.read_text(encoding='utf-8')
_old = 'torch.load(pth, map_location=device)'
_new = 'torch.load(pth, map_location=device, weights_only=False)'
if _old in _txt and 'weights_only=False' not in _txt:
    _finetune_py.write_text(_txt.replace(_old, _new), encoding='utf-8')
    print('Patched finetune_model.py for PyTorch 2.6+')

ECGFOUNDER_CHECKPOINT = Path(
    hf_hub_download(repo_id=ECGFOUNDER_HF_REPO, filename='12_lead_ECGFounder.pth')
)
print('ECGFounder root     :', ECGFOUNDER_ROOT)
print('ECGFounder checkpoint:', ECGFOUNDER_CHECKPOINT)


## Shared protocol

`notebooks/ecg_common.py` is imported. Checkpoint / early stop: **validation macro-AUPRC**. Architecture stays in this notebook; splits, metrics, thresholds, and the Trainer live in the shared module.


In [ ]:
from __future__ import annotations

import sys

import numpy as np
import pandas as pd
import torch
from torch import nn

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    CLASSES,
    EARLY_STOPPING_PATIENCE,
    EVALUATE_TEST as DEFAULT_EVALUATE_TEST,
    ExperimentConfig,
    MAX_EPOCHS,
    PRIMARY_MONITOR,
    SEEDS,
    build_trainer,
    finalize_run_config,
)

EVALUATE_TEST = DEFAULT_EVALUATE_TEST  # True: score test fold 10 with val-tuned thresholds
ACTIVE_SEEDS = [42]  # Final paper runs: ACTIVE_SEEDS = SEEDS
print('monitor=', PRIMARY_MONITOR, '| epochs=', MAX_EPOCHS, '| patience=', EARLY_STOPPING_PATIENCE)
print('ACTIVE_SEEDS', ACTIVE_SEEDS, '| EVALUATE_TEST', EVALUATE_TEST)
print('SEEDS (paper)', SEEDS, '| DEFAULT_EVALUATE_TEST', DEFAULT_EVALUATE_TEST)


## Model

`ft_12lead_ECGFounder(..., linear_prob=False)` loads pretrained Net1D, drops the 150-class head, and adds a new 5-class `dense` layer. The backbone is fully trainable.

`ECGFounder500HzWrapper` feeds native `(B, 12, 5000)` through a per-record global z-score. **No interpolation.** Keep `dropout` / architecture flags inside `model_factory` and `experiment.extra` together; changing only one side does not update the model.


In [ ]:
# MODEL_ARCHITECTURE
import torch
from torch import nn
from finetune_model import ft_12lead_ECGFounder


class ECGFounder500HzWrapper(nn.Module):
    """Native 500 Hz (B, 12, 5000) to pretrained ECGFounder. No interpolation."""

    def __init__(self, pretrained: nn.Module) -> None:
        super().__init__()
        self.pretrained = pretrained

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.ndim == 3 and tuple(x.shape[1:]) == (12, 5000), (
            f'expected (B, 12, 5000), got {tuple(x.shape)}'
        )
        mean = x.mean(dim=(1, 2), keepdim=True)
        std = x.std(dim=(1, 2), keepdim=True, unbiased=False).clamp_min(1e-8)
        x = (x - mean) / std
        return self.pretrained(x)


def _is_dense_head_param(name: str) -> bool:
    return name.endswith('dense.weight') or name.endswith('dense.bias')


def _extract_state_dict(payload):
    if not isinstance(payload, dict):
        return payload
    for key in ('state_dict', 'model_state_dict', 'model', 'net'):
        inner = payload.get(key)
        if isinstance(inner, dict) and inner and all(hasattr(v, 'shape') for v in list(inner.values())[:4]):
            payload = inner
            break
    return {
        (k[len('module.'):] if k.startswith('module.') else k): v
        for k, v in payload.items()
        if hasattr(v, 'shape')
    }


def report_pretrained_checkpoint(pretrained: nn.Module, ckpt_path) -> None:
    payload = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state = _extract_state_dict(payload)
    backbone_state = {
        k: v for k, v in state.items()
        if not (k == 'dense.weight' or k == 'dense.bias' or k.startswith('dense.'))
    }
    incompatible = pretrained.load_state_dict(backbone_state, strict=False)
    missing = list(getattr(incompatible, 'missing_keys', []))
    unexpected = list(getattr(incompatible, 'unexpected_keys', []))
    model_keys = [k for k in pretrained.state_dict() if not (k == 'dense.weight' or k == 'dense.bias' or k.startswith('dense.'))]
    loaded = [k for k in model_keys if k in backbone_state and tuple(backbone_state[k].shape) == tuple(pretrained.state_dict()[k].shape)]
    coverage = 100.0 * len(loaded) / max(len(model_keys), 1)
    print('Pretrained checkpoint:', ckpt_path)
    print('Missing keys:', missing)
    print('Unexpected keys:', unexpected)
    print(f'Pretrained backbone load coverage: {coverage:.2f} %')
    assert set(missing) == {'dense.weight', 'dense.bias'}, f'expected only dense head missing, got {missing}'
    assert unexpected == [], f'unexpected keys: {unexpected}'
    assert abs(coverage - 100.0) < 1e-6, f'backbone coverage {coverage:.2f} % != 100%'


def model_factory(num_classes: int) -> nn.Module:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pretrained = ft_12lead_ECGFounder(
        device=device,
        pth=str(ECGFOUNDER_CHECKPOINT),
        n_classes=num_classes,
        linear_prob=False,
    )
    for param in pretrained.parameters():
        param.requires_grad = True
    report_pretrained_checkpoint(pretrained, ECGFOUNDER_CHECKPOINT)
    return ECGFounder500HzWrapper(pretrained)


print('ECGFounder model factory ready | native 500 Hz | z-score only | full fine-tune')


## Training hyperparameters

Hyperparameters are defined in `ExperimentConfig(...)` in the next cell.

ECGFounder native 500 Hz: `batch_size=16`, `learning_rate=1e-4`. Shared protocol: `MAX_EPOCHS=50`, `EARLY_STOPPING_PATIENCE=7`, checkpoint = validation **macro-AUPRC**. `ACTIVE_SEEDS = [42]` is a single seed; for the paper use `ACTIVE_SEEDS = SEEDS` (`[42, 52, 62]`).

Checkpoints and thresholds are fit on val fold 9; reported performance analysis uses test fold 10.


In [ ]:
experiment = ExperimentConfig(
    model_name=MODEL_NAME,
    processed_dir=str(PROCESSED_DIR),
    output_root=str(EXPERIMENTS_DIR),
    learning_rate=1e-4,
    weight_decay=1e-5,
    epochs=MAX_EPOCHS,
    batch_size=16,
    sampling_rate=SAMPLING_RATE,
    seed=42,
    patience=EARLY_STOPPING_PATIENCE,
    monitor=PRIMARY_MONITOR,
    evaluate_test=EVALUATE_TEST,
    results_root=str(RESULTS_DIR),
    extra={
        'foundation_backend': FOUNDATION_BACKEND,
        'ecgfounder_hf_repo': ECGFOUNDER_HF_REPO,
        'ecgfounder_input': 'filename_hr_500hz',
        'linear_probe': False,
    },
)

print('config:', experiment.model_name, '| lr', experiment.learning_rate, '| batch', experiment.batch_size, '| epochs', experiment.epochs)
print('processed_dir', experiment.processed_dir, '| sampling_rate', experiment.sampling_rate)
print('X_train', tuple(np.load(Path(experiment.processed_dir) / 'X_train.npy', mmap_mode='r').shape))


In [ ]:
def _warn_existing_run(seed: int) -> None:
    run_dir = EXPERIMENTS_DIR / MODEL_NAME / f'seed_{int(seed)}'
    ckpt_dir = run_dir / 'checkpoints'
    if (ckpt_dir / 'last.pt').exists() or (ckpt_dir / 'best.pt').exists():
        print('=' * 72)
        print('Existing ecgfounder_500 run detected. This cell starts fresh (resume_from_last=False).')
        print(f'Found checkpoints under {ckpt_dir} — they will not be preloaded.')
        print('Use the resume cell to continue from last.pt.')
        print('=' * 72)


def _run_pretrain_sanity(trainer, data_module) -> None:
    processed = Path(experiment.processed_dir)
    X_train = np.load(processed / 'X_train.npy', mmap_mode='r')
    y_train = np.load(processed / 'y_train.npy', mmap_mode='r')
    ids_train = np.load(processed / 'ids_train.npy', mmap_mode='r')
    X_val = np.load(processed / 'X_val.npy', mmap_mode='r')
    y_val = np.load(processed / 'y_val.npy', mmap_mode='r')
    ids_val = np.load(processed / 'ids_val.npy', mmap_mode='r')

    print('Source sampling rate: 500 Hz (filename_hr / records500, cached by notebook 00)')
    print('Processed directory:', processed)
    print('X_train shape:', tuple(X_train.shape))
    print('y_train shape:', tuple(y_train.shape))
    print('ids_train shape:', tuple(ids_train.shape))
    assert X_train.ndim == 3 and tuple(X_train.shape[1:]) == (12, 5000)
    assert y_train.ndim == 2 and y_train.shape[1] == 5
    assert len(X_train) == len(y_train) == len(ids_train)
    assert len(X_val) == len(y_val) == len(ids_val)
    assert list(data_module.classes) == ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    assert CLASSES == ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    print('Class order:', list(data_module.classes))

    n_show = min(8, int(experiment.batch_size), len(X_train))
    raw = torch.from_numpy(np.asarray(X_train[:n_show], dtype=np.float32))
    print('raw batch shape:', tuple(raw.shape))
    assert tuple(raw.shape[1:]) == (12, 5000)
    x = raw.to(trainer.device)
    mean = x.mean(dim=(1, 2), keepdim=True)
    std = x.std(dim=(1, 2), keepdim=True, unbiased=False).clamp_min(1e-8)
    z = (x - mean) / std
    z_mean = z.mean(dim=(1, 2)).detach().cpu().numpy()
    z_std = z.std(dim=(1, 2), unbiased=False).detach().cpu().numpy()
    print('global z-score mean (records):', np.round(z_mean, 6))
    print('global z-score std  (records):', np.round(z_std, 6))
    assert np.allclose(z_mean, 0.0, atol=1e-5)
    assert np.allclose(z_std, 1.0, atol=1e-5)
    assert torch.isfinite(z).all() and not torch.isnan(z).any()

    wrapper = trainer.model
    backbone_trainable = 0
    head_trainable = 0
    frozen = 0
    total = 0
    for name, param in wrapper.named_parameters():
        n = int(param.numel())
        total += n
        if not param.requires_grad:
            frozen += n
            continue
        if _is_dense_head_param(name):
            head_trainable += n
        else:
            backbone_trainable += n
    print('Backbone trainable params:', backbone_trainable)
    print('Head trainable params:', head_trainable)
    print('Total trainable params:', backbone_trainable + head_trainable)
    print('Total model params:', total)
    assert backbone_trainable > 0, 'backbone must be trainable (full fine-tune)'
    assert head_trainable == 5125, f'head params expected 5125, got {head_trainable}'
    assert frozen == 0, f'all params must be trainable, frozen={frozen}'
    assert backbone_trainable + head_trainable == total
    with torch.no_grad():
        logits = wrapper(raw[:2].to(trainer.device))
    print('logits shape:', tuple(logits.shape))
    assert logits.shape[-1] == 5
    assert torch.isfinite(logits).all()


for seed in ACTIVE_SEEDS:
    experiment.seed = int(seed)
    _warn_existing_run(experiment.seed)
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)
    _run_pretrain_sanity(trainer, data_module)
    history_df = trainer.fit(
        data_module.train_dataloader(),
        data_module.val_dataloader(),
        epochs=experiment.epochs,
        resume_from_last=False,
    )
    protocol = trainer.evaluate_with_protocol(
        data_module.val_dataloader(),
        test_loader=data_module.test_dataloader(),
        evaluate_test=EVALUATE_TEST,
        results_root=RESULTS_DIR,
    )
    finalize_run_config(
        trainer,
        extra={
            'dataset_version': data_module.manifest.get('dataset_version') or data_module.manifest.get('raw_root_name'),
            'input': 'filename_hr_500hz',
            'ecgfounder_input': 'filename_hr_500hz',
            'sampling_rate': 500,
        },
    )
    print('best val macro-AUPRC', protocol.get('best_validation_macro_auprc'), 'epoch', protocol.get('best_epoch'))

history_df.tail()


## Resume from last.pt

If the `fit` cell above was interrupted, run this cell. It resumes **only** this native 500 Hz run under `experiments/ecgfounder_500/seed_42`. Interpolate checkpoints (`sr100_linear_interpolate_1000_to_5000`) are refused.


In [ ]:
import json

RESUME_RUN_DIR = None  # None: experiments/ecgfounder_500/seed_<seed>

if 'trainer' not in globals() or 'data_module' not in globals():
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)


def _assert_500hz_resume(run_dir) -> None:
    cfg_path = Path(run_dir) / 'config.json'
    if not cfg_path.exists():
        raise FileNotFoundError(f'config.json not found: {cfg_path}')
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    saved = str(cfg.get('model_name') or '')
    if saved and saved != experiment.model_name:
        raise ValueError(
            f'Refuse resume: {run_dir} belongs to {saved}, this notebook is {experiment.model_name}.'
        )
    if int(cfg.get('sampling_rate', -1)) != 500:
        raise ValueError(
            f'Refuse resume: sampling_rate={cfg.get("sampling_rate")} (need 500 Hz filename_hr run).'
        )
    if cfg.get('ecgfounder_input') == 'sr100_linear_interpolate_1000_to_5000':
        raise ValueError('Refuse resume: 100 Hz interpolate checkpoint.')
    if str(cfg.get('ecgfounder_input') or '') not in ('filename_hr_500hz', 'records500'):
        if int(cfg.get('sampling_rate', -1)) != 500:
            raise ValueError(f'Refuse resume: unexpected ecgfounder_input={cfg.get("ecgfounder_input")}')
    if cfg.get('linear_probe') or cfg.get('init_from_linear_probe_best'):
        raise ValueError('Refuse resume: linear-probe checkpoint.')


_resume_dir = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else (
    EXPERIMENTS_DIR / MODEL_NAME / f'seed_{int(experiment.seed)}'
)
_assert_500hz_resume(_resume_dir)
if not (_resume_dir / 'checkpoints' / 'last.pt').exists():
    raise FileNotFoundError(f'last.pt not found: {_resume_dir / "checkpoints" / "last.pt"}')

history_df = trainer.fit(
    data_module.train_dataloader(),
    data_module.val_dataloader(),
    epochs=experiment.epochs,
    resume_from_last=True,
    resume_run_dir=str(_resume_dir),
)
protocol = trainer.evaluate_with_protocol(
    data_module.val_dataloader(),
    test_loader=data_module.test_dataloader(),
    evaluate_test=EVALUATE_TEST,
    results_root=RESULTS_DIR,
)
finalize_run_config(
    trainer,
    extra={
        'dataset_version': data_module.manifest.get('dataset_version') or data_module.manifest.get('raw_root_name'),
        'input': 'filename_hr_500hz',
        'ecgfounder_input': 'filename_hr_500hz',
        'sampling_rate': 500,
    },
)
history_df.tail()


## Performance analysis

Loads `experiments/ecgfounder_500/seed_42/checkpoints/best.pt`. Thresholds are fit on val fold 9 (F1-max). Reported metrics, 2×2 confusion matrices, and prediction CSVs use **test fold 10**.

This cell runs standalone (pip / train cells above are not required). Files: `experiments/.../predictions/` and a copy under `results/ecgfounder/seed_42/` (canonical ensemble key).


In [ ]:
# ANALYZE_FROM_BEST_PT
# This cell runs standalone. Cells above (pip / imports / training) are not required.
import sys
import subprocess
from pathlib import Path

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'openpyxl',
        'scikit-learn',
        'pandas',
        'numpy',
        'matplotlib',
        'seaborn',
        'torch',
        'tqdm',
        'huggingface_hub',
    ]
)
try:
    from google.colab import drive

    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive') / 'ECG AI'
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    evaluate_and_display_best_pt,
    exec_notebook_cells,
    extra_from_run_config,
    load_json,
    seed_run_dir,
)

EVALUATE_TEST = True  # always score test fold 10 from best.pt
ANALYSIS_SEED = 42
MODEL_NAME = 'ecgfounder_500'
NOTEBOOK_PATH = PROJECT_DIR / 'notebooks' / '04_ecgfounder_500.ipynb'
run_dir = seed_run_dir(PROJECT_DIR / 'experiments', MODEL_NAME, ANALYSIS_SEED)
ckpt = run_dir / 'checkpoints' / 'best.pt'
if not ckpt.exists():
    raise FileNotFoundError(f'best.pt not found: {ckpt}')

cfg = load_json(run_dir / 'config.json') if (run_dir / 'config.json').exists() else {}
extra = extra_from_run_config(cfg)

import torch  # noqa: F401
from torch import nn  # noqa: F401

ECGFOUNDER_HF_REPO = str(extra.get('ecgfounder_hf_repo') or 'PKUDigitalHealth/ECGFounder')
exec_notebook_cells(NOTEBOOK_PATH, '# ECGFOUNDER_SETUP', globals())
exec_notebook_cells(NOTEBOOK_PATH, '# MODEL_ARCHITECTURE', globals())

def report_pretrained_checkpoint(*_a, **_k):
    return None  # analysis loads best.pt; skip HF coverage print

evaluate_and_display_best_pt(
    model_factory,
    run_dir,
    processed_dir=PROJECT_DIR / 'processed' / 'sr500',
    results_root=PROJECT_DIR / 'results',
    evaluate_test=True,
)
